<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install cuml
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 634, done.
remote: Counting objects: 100% (200/200), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 634 (delta 152), reused 90 (delta 88), pack-reused 434 (from 3)
Receiving objects: 100% (634/634), 209.09 KiB | 3.80 MiB/s, done.
Resolving deltas: 100% (326/326), done.
Installing RAPIDS remaining 26.02 libraries
Using Python 3.12.13 environment at: /usr
Resolved 176 packages in 1.95s
Prepared 11 packages in 1.17s
Uninstalled 5 packages in 164ms
Installed 11 packages in 54ms
 - bokeh==3.8.2
 + bokeh==3.6.3
 + cugraph-cu12==26.2.0
 + cuxfilter-cu12==26.2.0
 + datashader==0.19.0
 - holoviews==1.22.1
 + holoviews==1.20.2
 + jupyter-server-proxy==4.5.0
 - nvidia-cuda-nvcc-cu12==12.5.82
 + nvidia-cuda-nvcc-cu12==12.8.93
 - panel==1.8.10
 + panel==1.7.5
 + pyct==0.6.0
 - shapely==2.1.2
 + shapely==2.0.7
 + simpervisor==1.0.0

        ***********************************************************************

In [ ]:
# verify cuml installation
from cuml.ensemble import RandomForestRegressor as cuRF
print("cuML installed")

import cuml
print(f"cuML version: {cuml.__version__}")

cuML installed
cuML version: 26.02.000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import time
import gc
import warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from cuml.ensemble import RandomForestRegressor
import cupy as cp
# from sklearn.ensemble import RandomForestRegressor

In [ ]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
p_path     = "/content/drive/MyDrive/preprocess_data.parquet"
LOCAL_PATH = '/content/df_processed.parquet'

# preprocess and saved to disk
if not os.path.exists(LOCAL_PATH):
    df = pd.read_parquet(p_path)

    # preprocess
    id_cols    = ['DATE', 'permno']
    float_cols = [c for c in df.columns if c not in id_cols]
    df[float_cols] = df[float_cols].astype(np.float32)

    # generate dummy variables
    sic_df        = pd.get_dummies(df['sic2'], prefix='sic')
    sic_cols_list = sic_df.columns.tolist()
    df            = pd.concat([df, sic_df], axis=1)

    # saved to disk
    df.to_parquet(LOCAL_PATH, index=False)
    print(f"finished preprocessing, saved to local disk")

# skip preprocess
else:
    df            = pd.read_parquet(LOCAL_PATH)
    sic_cols_list = [c for c in df.columns if c.startswith('sic_')]
    print(f"read from local disk, skip preprocessing")

print(f"df shape: {df.shape}")

finished preprocessing, saved to local disk
df shape: (3712808, 183)


In [ ]:
def generate_920_features(df, char_cols, macro_cols, sic_cols):

    X_char = df[char_cols].to_numpy(dtype=np.float32, copy=False)
    X_macro = df[macro_cols].to_numpy(dtype=np.float32, copy=False)
    X_sic = df[sic_cols].to_numpy(dtype=np.float32, copy=False)

    # generate interaction (N x 752)

    X_inter = (X_char[:, :, np.newaxis] * X_macro[:, np.newaxis, :]).reshape(len(df), -1)

    # features (94) + interaction (752) + industry (74)
    X_920 = np.hstack([X_char, X_inter, X_sic])

    return X_920

In [ ]:
macro=['tbl','d/p','e/p','b/m','tms','dfy','ntis','svar']
features=list(df.columns)[2:96]

In [ ]:
def calc_oos_r2(actual, predicted):
    actual    = np.array(actual)
    predicted = np.array(predicted)
    denom = np.sum(actual ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((actual - predicted) ** 2) / denom

In [ ]:

def select_max_depth(X_train, y_train, X_val, y_val,
                     depth_candidates=[1, 2, 3, 4, 5, 6,7,8,9,10],
                     n_estimators=100,
                     prev_best_depth=None):

    # search    = depth_candidates   # search in the whole range
    if prev_best_depth is not None:
        # warm start
        idx = depth_candidates.index(prev_best_depth) \
              if prev_best_depth in depth_candidates else 0
        lo  = max(0, idx - 2)
        hi  = min(len(depth_candidates) - 1, idx + 2)
        search = depth_candidates[lo: hi + 1]
    else:
        search = depth_candidates

    X_tr = X_train.astype(np.float32)
    y_tr = y_train.astype(np.float32)
    X_v  = X_val.astype(np.float32)

    best_r2, best_depth = -np.inf, search[0]

    for depth in search:
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=depth,
            max_features='sqrt',
            bootstrap=True,
            random_state=42,
        )
        rf.fit(X_tr, y_tr)
        y_pred = rf.predict(X_v)

        # cuML get cupy array，turning back to numpy
        if hasattr(y_pred, 'get'):
            y_pred = y_pred.get()

        r2 = calc_oos_r2(y_val, y_pred)
        # print(f"    depth={depth}  val R²={r2*100:.4f}%")
        if r2 > best_r2:
            best_r2, best_depth = r2, depth

    return best_depth

In [ ]:
# def select_max_depth(X_train, y_train, X_val, y_val,
#                      depth_candidates=[1, 2, 3, 4, 5, 6],
#                      n_estimators=300,
#                      prev_best_depth=None):

#
#     search = depth_candidates

#     best_r2, best_depth = -np.inf, search[0]

#     for depth in search:
#         rf = RandomForestRegressor(
#             n_estimators=n_estimators,
#             max_depth=depth,
#             max_features='sqrt',
#             bootstrap=True,
#             n_jobs=-1,
#             random_state=42,
#         )
#         rf.fit(X_train, y_train)
#         r2 = calc_oos_r2(y_val, rf.predict(X_val))
#         # print(f"    depth={depth}  val R²={r2*100:.4f}%")
#         if r2 > best_r2:
#             best_r2, best_depth = r2, depth

#     return best_depth

In [ ]:
# def select_max_depth(X_train, y_train, X_val, y_val,
#                      depth_candidates=[1, 2, 3, 4, 5, 6],
#                      n_estimators=300,
#                      prev_best_depth=None):
#     """
#     optimize tree depth using validation set (OOS)

#     """
#     if prev_best_depth is not None:
#         # warm start
#         idx = depth_candidates.index(prev_best_depth) \
#               if prev_best_depth in depth_candidates else 0
#         lo  = max(0, idx - 1)
#         hi  = min(len(depth_candidates) - 1, idx + 1)
#         search = depth_candidates[lo: hi + 1]
#     else:
#         search = depth_candidates

#     best_r2, best_depth = -np.inf, search[0]

#     for depth in search:
#         rf = RandomForestRegressor(n_estimators=n_estimators,max_depth=depth,
#            max_features='sqrt',bootstrap=True,n_jobs=-1,
#             random_state=42,
#         )
#         rf.fit(X_train, y_train)
#         r2 = calc_oos_r2(y_val, rf.predict(X_val))
#         print(f"    depth={depth}  val R²={r2*100:.4f}%")
#         if r2 > best_r2:
#             best_r2, best_depth = r2, depth

#     return best_depth


In [ ]:
start_test_year    = 1987
end_test_year      = 2016
DEPTH_CANDIDATES   = [1, 2, 3, 4, 5, 6,7,8,9,10]
N_ESTIMATORS_SEL   = 150       # parameter selection on validation set
N_ESTIMATORS_FINAL = 500        # final model
# n_features = 920
# max_feat   = float(np.sqrt(n_features) / n_features)
all_preds      = []
prev_best_depth = None
dates_all = df['DATE'].values.copy()
y_all     = df['exret_lead1'].to_numpy(dtype=np.float32, copy=True)
#  winsorize
# from scipy.stats import mstats

# y_series = df['exret_lead1'].copy()
# lower = y_series.quantile(0.001)
# upper = y_series.quantile(0.999)
# y_series = y_series.clip(lower, upper)
# y_all = y_series.to_numpy(dtype=np.float32, copy=True)

# print(f"range after winsorize : [{y_all.min():.4f}, {y_all.max():.4f}]")
# print(f"s.d. after winsorize: {y_all.std():.6f}")
del df
gc.collect()

0

In [ ]:

needed_cols = features + macro + sic_cols_list + ['DATE', 'permno', 'exret_lead1', 'mvel1']
needed_cols = list(dict.fromkeys(needed_cols))

for year in range(start_test_year, end_test_year + 1):
    print(f"\n--- cope with {year} year ---")
    t0 = time.time()

    # split data
    train_mask = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
                 (dates_all <= pd.Timestamp(year - 13, 12, 31))
    val_mask   = (dates_all >= pd.Timestamp(year - 12, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year - 1,  12, 31))
    test_mask  = (dates_all >= pd.Timestamp(year, 1, 1)) & \
                 (dates_all <= pd.Timestamp(year, 12, 31))

    # filter
    df_year  = pq.read_table(LOCAL_PATH, columns=needed_cols).to_pandas()
    df_train = df_year[train_mask].reset_index(drop=True)
    df_val   = df_year[val_mask].reset_index(drop=True)
    df_test  = df_year[test_mask].reset_index(drop=True)
    del df_year
    gc.collect()

    # generate features
    X_train = generate_920_features(df_train, features, macro, sic_cols_list)
    y_train = y_all[train_mask]
    X_val   = generate_920_features(df_val,   features, macro, sic_cols_list)
    y_val   = y_all[val_mask]
    X_test  = generate_920_features(df_test,  features, macro, sic_cols_list)
    del df_train, df_val
    gc.collect()

    t1 = time.time()
    print(f'  feature construction: {t1-t0:.1f}s  '
          f'| train: {len(X_train):,}  val: {len(X_val):,}')

    # parameter selection

    best_depth = select_max_depth( X_train, y_train, X_val, y_val, depth_candidates=DEPTH_CANDIDATES,

      n_estimators=N_ESTIMATORS_SEL,prev_best_depth=prev_best_depth, )

    t2 = time.time()
    print(f"  best_depth={best_depth}  select parameter: {t2-t1:.1f}s")

    # combine train and validation set
    X_trainval = np.vstack([X_train, X_val])
    y_trainval = np.concatenate([y_train, y_val])
    del X_train, X_val, y_train, y_val
    gc.collect()

    final_model = RandomForestRegressor(
        n_estimators=N_ESTIMATORS_FINAL,
        max_depth=best_depth,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
    )
    final_model.fit(X_trainval, y_trainval)
    del X_trainval, y_trainval
    gc.collect()

    t3 = time.time()
    print(f"  train: {t3-t2:.1f}s  |  total: {t3-t0:.1f}s")

    # predict
    res = df_test[['DATE', 'permno', 'mvel1','exret_lead1']].copy().reset_index(drop=True)
    y_pred = final_model.predict(X_test.astype(np.float32))
    if hasattr(y_pred, 'get'):
      y_pred = y_pred.get()
    res['y_pred'] = y_pred
    del X_test, final_model, df_test
    gc.collect()

    all_preds.append(res)
    prev_best_depth = best_depth


# report

results = pd.concat(all_preds, ignore_index=True)

r2_all = calc_oos_r2(results['exret_lead1'], results['y_pred'])

top1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, False])
    .groupby('DATE', sort=False).head(1000)
)
r2_top = calc_oos_r2(top1000['exret_lead1'], top1000['y_pred'])

bot1000 = (
    results
    .sort_values(['DATE', 'mvel1'], ascending=[True, True])
    .groupby('DATE', sort=False).head(1000)
)
r2_bot = calc_oos_r2(bot1000['exret_lead1'], bot1000['y_pred'])

print(f"\n{'='*45}")
print(f"  {'Subsample':<25}  {'OOS R2':>10}")
print(f"{'-'*45}")
print(f"  {'All stocks':<25}  {r2_all*100:>+10.4f}%")
print(f"  {'Top 1000 (largest)':<25}  {r2_top*100:>+10.4f}%")
print(f"  {'Bottom 1000 (smallest)':<25}  {r2_bot*100:>+10.4f}%")
print(f"{'='*45}")




--- cope with 1987 year ---
  feature construction: 5.7s  | train: 472,278  val: 764,497
  best_depth=1  select parameter: 64.3s
  train: 41.2s  |  total: 111.2s

--- cope with 1988 year ---
  feature construction: 5.8s  | train: 530,435  val: 788,744
  best_depth=2  select parameter: 19.6s
  train: 41.0s  |  total: 66.4s

--- cope with 1989 year ---
  feature construction: 6.1s  | train: 588,534  val: 814,060
  best_depth=2  select parameter: 28.5s
  train: 44.6s  |  total: 79.3s

--- cope with 1990 year ---
  feature construction: 6.3s  | train: 647,363  val: 836,447
  best_depth=2  select parameter: 30.6s
  train: 47.9s  |  total: 84.9s

--- cope with 1991 year ---
  feature construction: 6.6s  | train: 704,916  val: 859,101
  best_depth=2  select parameter: 32.1s
  train: 49.6s  |  total: 88.3s

--- cope with 1992 year ---
  feature construction: 6.8s  | train: 761,970  val: 881,321
  best_depth=3  select parameter: 34.6s
  train: 52.1s  |  total: 93.6s

--- cope with 1993 year --

In [ ]:
# export
export_path = '/content/drive/MyDrive/rf_predictions1.parquet'
results.to_parquet(export_path, index=False)
print(f"to Google Drive: {export_path}")

to Google Drive: /content/drive/MyDrive/rf_predictions1.parquet


In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
# import cuml
# help(cuml.ensemble.RandomForestRegressor)

In [ ]:
# only 1987
df_year  = pq.read_table(LOCAL_PATH, columns=needed_cols).to_pandas()

train_mask_87 = (dates_all >= pd.Timestamp(1957, 3, 1)) & \
                (dates_all <= pd.Timestamp(1974, 12, 31))
val_mask_87   = (dates_all >= pd.Timestamp(1975, 1, 1)) & \
                (dates_all <= pd.Timestamp(1986, 12, 31))
test_mask_87  = (dates_all >= pd.Timestamp(1987, 1, 1)) & \
                (dates_all <= pd.Timestamp(1987, 12, 31))

X_train_87 = generate_920_features(df_year[train_mask_87], features, macro, sic_cols_list)
y_train_87 = y_all[train_mask_87]
X_val_87   = generate_920_features(df_year[val_mask_87],   features, macro, sic_cols_list)
y_val_87   = y_all[val_mask_87]
X_test_87  = generate_920_features(df_year[test_mask_87],  features, macro, sic_cols_list)
y_test_87  = y_all[test_mask_87]

print(f"train set: {X_train_87.shape}  y_mean={y_train_87.mean():.5f}  y_s.d.={y_train_87.std():.5f}")
print(f"validation set: {X_val_87.shape}    y_mean={y_val_87.mean():.5f}  y_s.d.={y_val_87.std():.5f}")
print(f"test set: {X_test_87.shape}   y_mean={y_test_87.mean():.5f}  y_s.d.={y_test_87.std():.5f}")


for depth in [1, 2, 3, 4, 5, 6]:
    from cuml.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=depth,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
    )
    rf.fit(X_train_87.astype(np.float32), y_train_87.astype(np.float32))

    y_pred_val  = rf.predict(X_val_87.astype(np.float32))
    y_pred_test = rf.predict(X_test_87.astype(np.float32))
    if hasattr(y_pred_val,  'get'): y_pred_val  = y_pred_val.get()
    if hasattr(y_pred_test, 'get'): y_pred_test = y_pred_test.get()

    r2_val  = calc_oos_r2(y_val_87,  y_pred_val)
    r2_test = calc_oos_r2(y_test_87, y_pred_test)
    print(f"  depth={depth}  val R²={r2_val*100:+.4f}%  test R²={r2_test*100:+.4f}%")